In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
travels_df = pd.read_csv('../data/raw/chegadas_1989.csv', encoding='latin1', sep=';')

In [3]:
travels_df.head()

,Continente,Ordem continente,País,Ordem país,UF,Ordem UF,Via de acesso,Ordem via de acesso,ano,Mês,Ordem mês,Chegadas
0,África,1,África do Sul,2,Amazonas,4,Aérea,1,1989,janeiro,1,9.0
1,África,1,Angola,6,Amazonas,4,Aérea,1,1989,janeiro,1,0.0
2,África,1,Nigéria,162,Amazonas,4,Aérea,1,1989,janeiro,1,0.0
3,África,1,Outros países,998,Amazonas,4,Aérea,1,1989,janeiro,1,0.0
4,América Central e Caribe,2,Costa Rica,53,Amazonas,4,Aérea,1,1989,janeiro,1,6.0


In [4]:
travels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17052 entries, 0 to 17051
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Continente           17052 non-null  str    
 1   Ordem continente     17052 non-null  int64  
 2   País                 17052 non-null  str    
 3   Ordem país           17052 non-null  int64  
 4   UF                   17052 non-null  str    
 5   Ordem UF             17052 non-null  int64  
 6   Via de acesso        17052 non-null  str    
 7   Ordem via de acesso  17052 non-null  int64  
 8   ano                  17052 non-null  int64  
 9   Mês                  17052 non-null  str    
 10  Ordem mês            17052 non-null  int64  
 11  Chegadas             16464 non-null  float64
dtypes: float64(1), int64(6), str(5)
memory usage: 1.6 MB


- travels_df.info() allows me to check in a quick look all my data, I have 17052 lines, a total of 12 columns
- It's interesting to see that I have for most of columns all of the data, we can check that on the non-null count column
we have the same amount of lines as non-null count, means no data is missing for them, but for "Chegadas" we have 16464 non null
which means 588 null lines. We need to check why later.
travels_df[travels_df['Chegadas'].isnull()]

In [5]:
# Removing some of the columns I won't be using, so I can reduce the amount of data, and increase visibility.
cols_remover = ['Ordem continente', 'Ordem país', 'Ordem UF', 'Ordem via de acesso']
travels_df = travels_df.drop(columns=cols_remover)

In [6]:
travels_df.head()

,Continente,País,UF,Via de acesso,ano,Mês,Ordem mês,Chegadas
0,África,África do Sul,Amazonas,Aérea,1989,janeiro,1,9.0
1,África,Angola,Amazonas,Aérea,1989,janeiro,1,0.0
2,África,Nigéria,Amazonas,Aérea,1989,janeiro,1,0.0
3,África,Outros países,Amazonas,Aérea,1989,janeiro,1,0.0
4,América Central e Caribe,Costa Rica,Amazonas,Aérea,1989,janeiro,1,6.0


In [7]:
# Isolate the data where 'Chegadas' == null, to figure out what's going on
null_df = travels_df[travels_df['Chegadas'].isnull()]

In [8]:
null_df.head()

,Continente,País,UF,Via de acesso,ano,Mês,Ordem mês,Chegadas
16464,África,África do Sul,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16465,África,Angola,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16466,África,Nigéria,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16467,África,Outros países,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16468,América Central e Caribe,Costa Rica,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN


In [9]:
# First 5 camps shows the same 'UF', 'Via de acesso', and 'Mês'. Are all null values associated to them?
null_df['UF'].value_counts()

UF
Mato Grosso do Sul    588
Name: count, dtype: int64

In [10]:
null_df['Via de acesso'].value_counts()

Via de acesso
Fluvial    588
Name: count, dtype: int64

In [11]:
null_df['Mês'].value_counts()

Mês
janeiro      49
fevereiro    49
março        49
abril        49
maio         49
junho        49
julho        49
agosto       49
setembro     49
outubro      49
novembro     49
dezembro     49
Name: count, dtype: int64

In [12]:
# In conclusion it seems 'Mato Grosso do Sul' had some kind of problem registering travel records when it was fluvial.
# Next step is trying to figure out why every month has count == 49

null_itens = null_df[null_df['Mês'] == 'janeiro']['País'].unique()
null_itens

<StringArray>
[           'África do Sul',                   'Angola',
                  'Nigéria',            'Outros países',
               'Costa Rica',                   'Panamá',
               'Porto Rico',                   'Canadá',
           'Estados Unidos',                   'México',
                'Argentina',                  'Bolívia',
                    'Chile',                 'Colômbia',
                  'Equador',          'Guiana Francesa',
                   'Guiana',                 'Paraguai',
                     'Peru',                 'Suriname',
                  'Uruguai',                'Venezuela',
                    'China',      'República da Coreia',
                    'Japão',                 'Alemanha',
                  'Áustria',                  'Bélgica',
                'Dinamarca',                  'Espanha',
                   'França',                   'Grécia',
                  'Holanda',              'Reino Unido',
                 

In [13]:
# Ok, so 45 different countries, but it shows 49 every month.

null_january = null_df[null_df['Mês'] == 'janeiro']
null_january.shape

(49, 8)

In [14]:
null_january

,Continente,País,UF,Via de acesso,ano,Mês,Ordem mês,Chegadas
16464,África,África do Sul,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16465,África,Angola,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16466,África,Nigéria,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16467,África,Outros países,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16468,América Central e Caribe,Costa Rica,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16469,América Central e Caribe,Panamá,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16470,América Central e Caribe,Porto Rico,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16471,América Central e Caribe,Outros países,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16472,América do Norte,Canadá,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN
16473,América do Norte,Estados Unidos,Mato Grosso do Sul,Fluvial,1989,janeiro,1,NaN


In [15]:
# Now it's clear what is causing the confusion about the data, 'Outros países' repeats once for all continents except Oceania.

# Handling Missing Values
# Based on our analysis, the 588 null values are restricted to 'Fluvial' access in 'Mato Grosso do Sul'.
# Filling these null values with 0, assuming no arrivals were recorded.
travels_df['Chegadas'] = travels_df['Chegadas'].fillna(0)

In [16]:
# Checking if it worked
travels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17052 entries, 0 to 17051
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Continente     17052 non-null  str    
 1   País           17052 non-null  str    
 2   UF             17052 non-null  str    
 3   Via de acesso  17052 non-null  str    
 4   ano            17052 non-null  int64  
 5   Mês            17052 non-null  str    
 6   Ordem mês      17052 non-null  int64  
 7   Chegadas       17052 non-null  float64
dtypes: float64(1), int64(2), str(5)
memory usage: 1.0 MB


In [17]:
# Once there are no more null values,'Chegadas' data type doesn't need to be float anymore, converting it to integer (you can't have half a person)
travels_df['Chegadas'] = travels_df['Chegadas'].astype(int)

In [18]:
# Trimming whitespace from all string columns to avoid grouping errors

string_columns = ['Continente', 'País', 'UF', 'Via de acesso', 'Mês']
for col in string_columns:
    travels_df[col] = travels_df[col].str.strip()

In [19]:
# Checking the results
print("Final Data Types:")
print(travels_df.dtypes)

Final Data Types:
Continente         str
País               str
UF                 str
Via de acesso      str
ano              int64
Mês                str
Ordem mês        int64
Chegadas         int64
dtype: object
